In [1]:
""" 
    1. class Module_DNN
        학습 모델 제작
    
    2. class Module_TT
        training()  train
        evaluate()  test용 함수.
"""

' \n    1. class Module_DNN\n        학습 모델 제작\n    \n    2. class Module_TT\n        training()  train\n        evaluate()  test용 함수.\n'

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import torchinfo

In [ ]:
# 은닉층 개수 동적인 모델 ---------------------------------------------------------------------
class MyModel(nn.Module):
    def __init__(self, in_in, out_out, h_in, h_list=[]):
        """ 
        in_in : 입력층의 입력. 피처 개수
        h_in : 입력층 출력. 최초 히든입력.
        out_out : 최종 출력값.
        h_list = [] 리스트
        """
        # 부모클래스 생성
        super().__init__()
        # 자식클래스의 인스턴스 속성 설정
        self.input_layer = nn.Linear(in_in, h_in)
        
        self.h1_layer = nn.ModuleDict() 
        for idx in range(len(h_list)):
            h_in = h_list[idx-1] if idx else h_in
            h_out = h_list[idx]
            self.h1_layer[f"hl_{str(idx)}"] = nn.Linear(h_in,h_out)
            
        self.output_layer = nn.Linear(h_out, out_out)
        
    def forward(self, x):
        y=F.relu(self.input_layer(x))
    
        for linear in self.h1_layer.values():
            y=F.relu(linear(y))
            
        return self.output_layer(y)

In [ ]:
class MyModel2(nn.Module):
    def __init__(self, in_in, out_out, h_list=[]):
        """ 
        in_in : 입력층 입력 차원
        out_out : 출력층 출력 차원
        h_list : [(노드 개수, 'layer_type')] 형태의 리스트 (입력층 포함)
        """
        super().__init__()

        # 첫 번째 레이어를 입력층으로 설정
        first_out, first_layer_type = h_list[0]
        
        if first_layer_type == 'linear':
            self.input_layer = nn.Linear(in_in, first_out)
        elif first_layer_type == 'conv':  
            self.input_layer = nn.Conv2d(in_in, first_out, kernel_size=3, stride=1, padding=1)
        elif first_layer_type == 'lstm':  
            self.input_layer = nn.LSTM(in_in, first_out, batch_first=True)
        elif first_layer_type == 'transformer':  
            self.input_layer = nn.TransformerEncoderLayer(d_model=in_in, nhead=4)
        else:
            raise ValueError(f"지원하지 않는 입력층 타입: {first_layer_type}")

        # 은닉층 설정
        self.h1_layer = nn.ModuleDict()
        prev_dim = first_out

        for idx, (h_out, layer_type) in enumerate(h_list[1:]):  # 첫 번째 레이어는 입력층으로 설정했으므로 제외
            if layer_type == 'linear':
                self.h1_layer[f"hl_{idx}"] = nn.Linear(prev_dim, h_out)
            elif layer_type == 'conv':
                self.h1_layer[f"hl_{idx}"] = nn.Conv2d(prev_dim, h_out, kernel_size=3, stride=1, padding=1)
            elif layer_type == 'lstm':
                self.h1_layer[f"hl_{idx}"] = nn.LSTM(prev_dim, h_out, batch_first=True)
            elif layer_type == 'transformer':
                self.h1_layer[f"hl_{idx}"] = nn.TransformerEncoderLayer(d_model=prev_dim, nhead=4)
            else:
                raise ValueError(f"지원하지 않는 레이어 타입: {layer_type}")

            prev_dim = h_out
        
        # 출력층
        self.output_layer = nn.Linear(prev_dim, out_out)

    def forward(self, x):
        if isinstance(self.input_layer, nn.LSTM):
            x, _ = self.input_layer(x)  
        else:
            x = F.relu(self.input_layer(x))  

        for layer in self.h1_layer.values():
            if isinstance(layer, nn.LSTM):
                x, _ = layer(x)
            else:
                x = F.relu(layer(x))

        return self.output_layer(x)

